# Overfitting

## Learning Objectives
1. Demonstrate polynomial degree vs train/val loss (classic overfitting) using NumPy.
2. Plot learning curves and validation curves with scikit-learn.
3. Apply a regularisation ladder (L1, L2, Dropout, Early Stopping) to reduce overfitting.
4. Visualise the bias-variance tradeoff and dataset size effect.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    learning_curve, validation_curve, cross_val_score
)
from sklearn.datasets import make_regression, make_classification

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Polynomial Degree vs Overfitting (NumPy)


In [ ]:
n_train, n_val = 40, 200
x_all = np.linspace(0, 1, n_train + n_val)
np.random.shuffle(x_all)
x_train, x_val = x_all[:n_train], x_all[n_train:]
y_fn = lambda x: np.sin(2 * np.pi * x) + 0.2 * x ** 2
y_train = y_fn(x_train) + np.random.randn(n_train) * 0.2
y_val   = y_fn(x_val)   + np.random.randn(n_val)   * 0.2

degrees = list(range(1, 16))
train_rmse_list, val_rmse_list = [], []

for deg in degrees:
    X_tr_poly = np.stack([x_train ** d for d in range(deg + 1)], axis=1)
    X_va_poly = np.stack([x_val   ** d for d in range(deg + 1)], axis=1)
    coeffs, *_ = np.linalg.lstsq(X_tr_poly, y_train, rcond=None)
    y_tr_pred = X_tr_poly @ coeffs
    y_va_pred = X_va_poly @ coeffs
    train_rmse_list.append(np.sqrt(np.mean((y_train - y_tr_pred) ** 2)))
    val_rmse_list.append(np.sqrt(np.mean((y_val   - y_va_pred) ** 2)))

best_deg = degrees[np.argmin(val_rmse_list)]
print(f"Best degree (min val RMSE): {best_deg}")
print(f"{'Degree':>7} | {'Train RMSE':>11} | {'Val RMSE':>10}")
print("-" * 34)
for d, tr, va in zip(degrees, train_rmse_list, val_rmse_list):
    flag = " <-- optimal" if d == best_deg else ""
    print(f"{d:>7d} | {tr:>11.4f} | {va:>10.4f}{flag}")


## Level 2: Learning Curves and Validation Curves (sklearn)


In [ ]:
X_lc, y_lc = make_regression(n_samples=600, n_features=20, noise=15, random_state=42)
X_lc = StandardScaler().fit_transform(X_lc)

pipe_lc = Pipeline([("ridge", Ridge(alpha=1.0))])
train_sizes, train_scores_lc, val_scores_lc = learning_curve(
    pipe_lc, X_lc, y_lc,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
)
train_rmse_lc = -train_scores_lc.mean(axis=1)
val_rmse_lc   = -val_scores_lc.mean(axis=1)

alphas = np.logspace(-4, 4, 20)
train_scores_vc, val_scores_vc = validation_curve(
    Ridge(), X_lc, y_lc,
    param_name="alpha", param_range=alphas,
    cv=5, scoring="neg_root_mean_squared_error"
)
best_alpha_idx = np.argmin(-val_scores_vc.mean(axis=1))
print(f"Learning curve: min val RMSE = {val_rmse_lc.min():.4f} "
      f"at n_train = {train_sizes[np.argmin(val_rmse_lc)]:.0f}")
print(f"Best Ridge alpha (val curve): {alphas[best_alpha_idx]:.4f}")
print(f"Min val RMSE at best alpha  : "
      f"{-val_scores_vc.mean(axis=1)[best_alpha_idx]:.4f}")
# Note: for GPU-backed models, wrap in try/except RuntimeError to catch
# out of memory errors: if OOM, reduce n_samples or batch_size.


## Real-World Example 1: Regularisation Ladder


In [ ]:
X_reg, y_reg = make_classification(n_samples=200, n_features=50, n_informative=10,
                                    n_redundant=20, random_state=42)
scaler_reg = StandardScaler().fit(X_reg)
X_reg = scaler_reg.transform(X_reg)
X_reg_t = torch.FloatTensor(X_reg[:160]); y_reg_t = torch.LongTensor(y_reg[:160])
X_reg_v = torch.FloatTensor(X_reg[160:]).to(device)
y_reg_v = torch.LongTensor(y_reg[160:]).to(device)
reg_loader = DataLoader(TensorDataset(X_reg_t, y_reg_t), batch_size=32, shuffle=True)


def train_regularised(l2_wd=0.0, use_dropout=False, use_early_stop=False,
                       n_epochs=100):
    layers = [nn.Linear(50, 128), nn.ReLU()]
    if use_dropout: layers.append(nn.Dropout(0.5))
    layers += [nn.Linear(128, 64), nn.ReLU()]
    if use_dropout: layers.append(nn.Dropout(0.5))
    layers.append(nn.Linear(64, 2))
    m = nn.Sequential(*layers).to(device)
    o = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=l2_wd)
    c = nn.CrossEntropyLoss()
    best_val_acc, patience, wait = 0.0, 10, 0

    for epoch in range(n_epochs):
        m.train()
        for xb, yb in reg_loader:
            o.zero_grad(); c(m(xb.to(device)), yb.to(device)).backward(); o.step()
        m.eval()
        with torch.no_grad():
            val_acc = (m(X_reg_v).argmax(1) == y_reg_v).float().mean().item()
            tr_acc  = (m(X_reg_t.to(device)).argmax(1)
                       == y_reg_t.to(device)).float().mean().item()
        if val_acc > best_val_acc:
            best_val_acc = val_acc; wait = 0
        else:
            wait += 1
        if use_early_stop and wait >= patience:
            break
    return tr_acc, best_val_acc


configs = [
    ("No regularisation",       dict(l2_wd=0.0,  use_dropout=False, use_early_stop=False)),
    ("L2 (wd=0.01)",            dict(l2_wd=0.01, use_dropout=False, use_early_stop=False)),
    ("L2 + Dropout",            dict(l2_wd=0.01, use_dropout=True,  use_early_stop=False)),
    ("L2+Dropout+EarlyStop",    dict(l2_wd=0.01, use_dropout=True,  use_early_stop=True)),
]
print(f"{'Configuration':>30} | {'Train Acc':>10} | {'Val Acc':>9} | {'Gap':>6}")
print("-" * 64)
for name, kwargs in configs:
    tr_a, va_a = train_regularised(**kwargs)
    print(f"{name:>30} | {tr_a:>10.4f} | {va_a:>9.4f} | {tr_a-va_a:>6.4f}")


## Real-World Example 2: Dataset Size Effect on Overfitting


In [ ]:
def overfit_with_n(n_train: int, n_features: int = 100, n_epochs: int = 50):
    """Train on n_train samples and return (train_acc, val_acc)."""
    X_s, y_s = make_classification(n_samples=n_train + 200, n_features=n_features,
                                    n_informative=10, n_redundant=30, random_state=0)
    sc = StandardScaler().fit(X_s[:n_train])
    X_s = sc.transform(X_s)
    X_tr = torch.FloatTensor(X_s[:n_train]); y_tr = torch.LongTensor(y_s[:n_train])
    X_va = torch.FloatTensor(X_s[n_train:]).to(device)
    y_va = torch.LongTensor(y_s[n_train:]).to(device)
    loader_s = DataLoader(TensorDataset(X_tr, y_tr),
                          batch_size=min(32, n_train), shuffle=True)
    m = nn.Sequential(
        nn.Linear(n_features, 128), nn.ReLU(),
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 2)).to(device)
    o = torch.optim.Adam(m.parameters(), lr=1e-3)
    c = nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        m.train()
        for xb, yb in loader_s:
            o.zero_grad(); c(m(xb.to(device)), yb.to(device)).backward(); o.step()
    m.eval()
    with torch.no_grad():
        tr_acc = (m(X_tr.to(device)).argmax(1)==y_tr.to(device)).float().mean().item()
        va_acc = (m(X_va).argmax(1)==y_va).float().mean().item()
    return tr_acc, va_acc


sample_sizes = [50, 100, 200, 400, 800, 1600]
print(f"{'n_train':>8} | {'Train Acc':>10} | {'Val Acc':>9} | {'Gap':>8}")
print("-" * 44)
for n in sample_sizes:
    tr, va = overfit_with_n(n)
    print(f"{n:>8} | {tr:>10.4f} | {va:>9.4f} | {tr-va:>8.4f}")


## Real-World Example 3: Bias-Variance Tradeoff Visualisation


In [ ]:
def bv_decompose(degree: int, n_trials: int = 40, n_train: int = 30):
    """Estimate bias^2 and variance for polynomial regression of given degree."""
    preds_at_x0 = []
    x0 = np.array([[0.5]])

    for _ in range(n_trials):
        x_tr = np.random.uniform(0, 1, n_train).reshape(-1, 1)
        y_tr = np.sin(2 * np.pi * x_tr.ravel()) + np.random.randn(n_train) * 0.2
        pipe_bv = Pipeline([
            ("poly", PolynomialFeatures(degree=degree, include_bias=True)),
            ("reg",  LinearRegression()),
        ])
        pipe_bv.fit(x_tr, y_tr)
        preds_at_x0.append(pipe_bv.predict(x0)[0])

    preds = np.array(preds_at_x0)
    y0_true = np.sin(2 * np.pi * 0.5)
    bias2 = (preds.mean() - y0_true) ** 2
    variance = preds.var()
    return bias2, variance


degrees_bv = list(range(1, 14))
b2s, vars_, totals = [], [], []
for deg in degrees_bv:
    b2, var = bv_decompose(deg, n_trials=60, n_train=30)
    b2s.append(b2); vars_.append(var); totals.append(b2 + var)
    print(f"Degree {deg:2d}: bias^2={b2:.4f}, variance={var:.4f}, total={b2+var:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(degrees_bv, b2s,    label="Bias^2",    color="steelblue",  linewidth=2)
ax.plot(degrees_bv, vars_,  label="Variance",  color="coral",      linewidth=2)
ax.plot(degrees_bv, totals, label="Total",     color="navy",
        linestyle="--", linewidth=2)
ax.axvline(x=degrees_bv[np.argmin(totals)], color="green",
           linestyle=":", label="Optimal")
ax.set_xlabel("Polynomial Degree"); ax.set_ylabel("Error")
ax.set_title("Bias-Variance Tradeoff"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/overfitting_bias_variance.png", dpi=80)
plt.close()
print("Bias-variance plot saved to /tmp/overfitting_bias_variance.png")
